# 5 pamoka – Agentinis RAG


## Sąranka

Ši užrašų knygelė demonstruoja Agentic RAG (Retrieval-Augmented Generation) modelį naudojant Microsoft Agent Framework.

**Būtini dalykai:**
- `AZURE_AI_PROJECT_ENDPOINT` — jūsų Microsoft Foundry projekto galinis taškas
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — jūsų modelio diegimo pavadinimas (pvz., `gpt-5-mini`)
- Azure CLI autentifikuotas (`az login`)

> **Pastaba:** Ši užrašų knygelė naudoja atminties bazę, todėl galite susitelkti ties pačiu agentic RAG modeliu — nereikia Azure AI Search šaltinio. Norėdami naudoti tą patį modelį su tikru Azure AI Search indeksu (kaip gamyboje), žr. pasirenkamą [Azure AI Search diegimo gido](../../00-course-setup/AzureSearch.md) skyrių.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kas yra Agentinis RAG?

Tradicinis RAG seka fiksuotą procesą: surinkti dokumentus, tada sugeneruoti atsakymą. **Agentinis RAG** žengia toliau, suteikdamas agentui autonomiją nuspręsti, **kada** ir **kaip** gauti informaciją.

Naudojant agentinį RAG, agentas gali:
- **Nuspręsti**, ar reikia paieškos prieš atsakant į klausimą
- **Pasirinkti**, kurį duomenų šaltinį ar įrankį naudoti užklausai
- **Įvertinti** gautus rezultatus ir atlikti papildomas paieškas, jei pirmasis bandymas nepakankamas
- **Sujungti** informaciją iš kelių paieškos etapų į nuoseklų atsakymą

Tai padaro agentą lankstesnį ir tikslesnį, palyginti su statiniu surink- ir-gamink procesų srautu.


## Paieškos įrankio kūrimas

Agentic RAG išoriniai duomenų šaltiniai yra apgaubti kaip **įrankiai**, kuriuos agentas gali iškviesti pagal poreikį. Tai leidžia agentui elgtis su paieška kaip su dar viena veikla, kurią jis gali atlikti, o ne kaip privalomu žingsniu.

Žemiau apibrėžiame kelionių žinių bazę ir ją pateikiame kaip įrankį, kurį agentas gali iškviesti norėdamas sužinoti informaciją apie kelionės tikslą.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG agente kūrimas

Dabar sukursime agentą, kuriam nurodyta **visada prieš atsakant rinkti informaciją**. Agentas naudoja įrankį `search_travel_knowledge`, kad pagrįstų savo atsakymus žinių baze, o ne remtųsi savo treniruočių duomenimis.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratyvus gavimas – Maker-Checker modelis

Vienas iš Agentic RAG esminių privalumų yra **iteratyvus gavimas**. Agentas gali atlikti kelis paieškos etapus, kad patikrintų, patobulintų ar išplėstų savo pradines išvadas – panašiai kaip „maker-checker“ darbo eiga:

1. **Maker žingsnis**: Agentas gauna pradinę informaciją ir sudaro atsakymo juodraštį.
2. **Checker žingsnis**: Agentas atlieka papildomas paieškas, kad patikrintų detales arba užpildytų spragas.

Žemiau agentui pateikiamas klausimas, reikalaujantis palyginti kelias paskirties vietas, todėl jis ieško kelis kartus.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Santrauka

Šioje pamokoje sužinojote, kaip sukurti **Agentic RAG** sistemą naudojant Microsoft Agent Framework:

- **Agentic RAG** leidžia agentams savarankiškai nuspręsti, kada vykdyti informacijos paiešką, todėl paieška tampa dinamiška, o ne fiksuota.
- **Įrankiai kaip duomenų šaltiniai**: Išorinės žinių bazės (pvz., Azure AI Search) yra įvyniotos į įrankius, kuriuos agentas gali paleisti.
- **Iteratyvi paieška**: maker-checker modelis leidžia agentui atlikti kelis paieškos ciklus — ieškoti, tikrinti ir tobulinti — prieš pateikiant galutinį atsakymą.

Naudojant gamyboje, reikėtų pakeisti atmintyje esantį `TRAVEL_KNOWLEDGE_BASE` į tikrą Azure AI Search indeksą, kad būtų galima tvarkyti didelio masto kelionių dokumentų paiešką.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
